<a href="https://colab.research.google.com/github/kjahan/unified_qa_demo/blob/main/notebooks/unified_qa_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers[sentencepiece]

     |████████████████████████████████| 3.4 MB 5.2 MB/s 
     |████████████████████████████████| 596 kB 54.0 MB/s 
     |████████████████████████████████| 895 kB 42.6 MB/s 
     |████████████████████████████████| 61 kB 242 kB/s 
     |████████████████████████████████| 3.3 MB 7.5 MB/s 
     |████████████████████████████████| 1.2 MB 64.3 MB/s 
  Attempting uninstall: pyyaml
    Found existing installation: PyYAML 3.13
    Uninstalling PyYAML-3.13:
      Successfully uninstalled PyYAML-3.13


In [3]:
from transformers import AutoTokenizer, T5ForConditionalGeneration

In [4]:
model_name = "allenai/unifiedqa-t5-small" # you can specify the model size here
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

Downloading:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/1.20k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/773k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/1.74k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/231M [00:00<?, ?B/s]

In [5]:
def run_model(input_string, **generator_args):
    input_ids = tokenizer.encode(input_string, return_tensors="pt")
    res = model.generate(input_ids, **generator_args)
    return tokenizer.batch_decode(res, skip_special_tokens=True)

In [6]:
run_model("which is best conductor? \\n (a) iron (b) feather")


['iron']

In [7]:
run_model("scott filled a tray with juice and put it in a freezer. the next day, scott opened the freezer. how did the juice most likely change? \\n (a) it condensed. (b) it evaporated. (c) it became a gas. (d) it became a solid.")


['it condensed.']

In [8]:
run_model("which is best conductor? \\n (a) iron (b) feather (c) wood (d) plastic",
         temperature=0.9, num_return_sequences=4, num_beams=20)


['iron', 'feather', 'iron (b) feather', 'iron (c) feather']

In [9]:
run_model('At what speed did the turbine operate? \n (Nikola_Tesla) On his 50th birthday in 1906, Tesla demonstrated his 200 horsepower (150 kilowatts) 16,000 rpm bladeless turbine')

['16,000 rpm']

In [10]:
run_model("What does a drink from narcissus's spring cause the drinker to do? \n Mercury has awakened Echo, who weeps for Narcissus, and states that a drink from Narcissus's spring causes the drinkers to ''Grow dotingly enamored of themselves.''")

['grow dotingly enamored of themselves']

In [11]:
run_model("What does photosynthesis produce that helps plants grow? \n (A) water (B) oxygen (C) protein (D) sugar")

['sugar']

In [12]:
run_model("Who was Billy? \n (A) The skinny kid (B) A teacher (C) A little kid (D) The big kid \n Billy was like a king on the school yard. A king without a queen. He was the biggest kid in our grade, so he made all the rules during recess.")

['The big kid Billy was like a king on the school yard. A king without']

In [13]:
run_model("Was America the first country to have a president? \n (President) The first usage of the word president to denote the highest official in a government was during the Commonwealth of England")

['no']

In [14]:
run_model('what is 7-2? \n (A) 1 (B) 32 (C) 5')

['5']

In [15]:
run_model('What is the type of roof? \n There is an older tar and gravel on the main roof, blisters, bared area and bubbles noted. There is no sign of roof leak attime of inspection however it is at the end of its serviceable life.')

['tar and gravel']

In [16]:
run_model("What is the type of foundation? \n Subject has a perimeter concrete foundation. Subject foundation is consistence to property of its age, settlement cracks and crumbling noted however it appears to be in serviceable condition and inspection is based on visual only. Owner is recommended to contact related trade for necessary correction.")



['perimeter concrete foundation']

In [17]:
run_model("What are the issues with the foundation? \n Subject has a perimeter concrete foundation. Subject foundation is consistence to property of its age, settlement cracks and crumbling noted however it appears to be in serviceable condition and inspection is based on visual only. Owner is recommended to contact related trade for necessary correction.")


['cracks and crumbling']

## Install pdf to text

https://stackoverflow.com/questions/25665/python-module-for-converting-pdf-to-text
https://www.unixuser.org/~euske/python/pdfminer/index.html
https://github.com/pdfminer/pdfminer.six

In [18]:
!pip install pdfminer.six

     |████████████████████████████████| 5.6 MB 4.5 MB/s 
     |████████████████████████████████| 3.6 MB 42.9 MB/s 


## pdfminer3

https://stackoverflow.com/questions/56494070/how-to-use-pdfminer-six-with-python-3

https://github.com/gwk/pdfminer3/

In [19]:
!pip install pdfminer3

     |████████████████████████████████| 5.0 MB 5.0 MB/s 
     |████████████████████████████████| 2.0 MB 45.4 MB/s 
  Created wheel for pdfminer3: filename=pdfminer3-2018.12.3.0-py3-none-any.whl size=117825 sha256=f0d5567b0e3b3dea2f033a65ebe40f8eae5b051e4e5e671f088149e3ccb79400
  Stored in directory: /root/.cache/pip/wheels/f6/1b/21/339d1825e274c4a9829233a986f93dcedb98913f98e85b2916
Successfully built pdfminer3


Upload `36. inspection report Home.pdf` to sample_data folder in your Colab!

In [20]:
pdf_fn = "sample_data/36. inspection report Home.pdf"

In [21]:
from pdfminer3.layout import LAParams, LTTextBox
from pdfminer3.pdfpage import PDFPage
from pdfminer3.pdfinterp import PDFResourceManager
from pdfminer3.pdfinterp import PDFPageInterpreter
from pdfminer3.converter import PDFPageAggregator
from pdfminer3.converter import TextConverter
import io

resource_manager = PDFResourceManager()
fake_file_handle = io.StringIO()
converter = TextConverter(resource_manager, fake_file_handle, laparams=LAParams())
page_interpreter = PDFPageInterpreter(resource_manager, converter)

with open(pdf_fn, 'rb') as fh:

    for page in PDFPage.get_pages(fh,
                                  caching=True,
                                  check_extractable=True):
        page_interpreter.process_page(page)

    text = fake_file_handle.getvalue()

# close open handles
converter.close()
fake_file_handle.close()

print(text)

Inspection Report

This inspection report
prepared specifically for:
Won Living Trust
1059 Montgomery Street
San Francisco, CA  94133

Inspected by:

Zon K. Chu

ZC & Associates
235 Westlake Center #381
Daly City, CA  94015
650-992-1191          650-504-3161

Table of Contents

General Information. . . .
Deficiency Summary
Roof. . . . . . . . . . . . . . .
Exterior. . . . . . . . . . . . .
Grounds & Drainage. . . .
Heating & Cooling. . . . .
Plumbing. . . . . . . . . . . .
Electrical. . . . . . . . . . . .

1
D1
2
3
4
5
6
8

Bathrooms. . . . . . . . . . . .
Interior Rooms. . . . . . . . .
Garage & Carport. . . . . .
Attic. . . . . . . . . . . . . . . . .
Foundation. . . . . . . . . . .

11
12
14
15
16

Copyright 2013 © • New Image Software, Inc. • All rights reserved. Rev 7/1/2013

Buyer Received 1 - 20 pagesSignDATESignDATEZC & Associates
235 Westlake Center #381
Daly City, CA  94015
650-992-1191          650-504-3161

Customer File #

Sellers Agnt

:

211122001
Michelle An

Seller



In [23]:
text.index("foundation")

9474

In [25]:
text[9474:10100]

'foundation. Subject foundation is consistence to property of its age, settlement cracks\nand crumbling noted however it appears to be in serviceable condition and inspection is based on visual only. Owner is\nrecommended to contact related trade for necessary correction.\n\nThe report is provided as a courtesy for quicker access to DEFICIENCIES within the inspection report. This is not intended as a substitute for reading the inspection report.\n\nItems listed may be discussed further on the corresponding report page.  There also may be findings other than what is listed on this page.\n\nD3\n\nCopyright 2013 © • New Image Softwa'

## Foundation

In [28]:
foundation_paragraph_1 = "Subject foundation is consistence to property of its age, settlement cracks\nand crumbling noted however it appears to be in serviceable condition and inspection is based on visual only. Owner is\nrecommended to contact related trade for necessary correction.\n\nThe report is provided as a courtesy for quicker access to DEFICIENCIES within the inspection report."
foundation_paragraph_2 = "Subject has a perimeter concrete foundation. Subject foundation is consistence to property of its age, settlement cracks and crumbling noted however it appears to be in serviceable condition and inspection is based on visual only. Owner is recommended to contact related trade for necessary correction."

run_model("Are ther cracks in the foundation? \n {}".format(foundation_paragraph_2))


['yes']

In [33]:
text.index("Roof Covering")

5185

In [38]:
roof_covering_pragraph_1 = text[5185:6000]

run_model("Is there any leak in the roof? \n {}".format(roof_covering_pragraph_1))


['no']

In [37]:
roof_covering_pragraph_1

'Roof Covering\nThere is an older tar and gravel on the main roof, blisters, bared area and bubbles noted. There is no sign of roof leak at\ntime of inspection however it is at the end of its serviceable life.\n\n13 - Roof Covering\n\nThe report is provided as a courtesy for quicker access to DEFICIENCIES within the inspection report. This is not intended as a substitute for reading the inspection report.\n\nItems listed may be discussed further on the corresponding report page.  There also may be findings other than what is listed on this page.\n\nD1\n\nCopyright 2013 © • New Image Software, Inc. • All rights reserved. Rev 7/1/2013\n\n\x0cZC & Associates\n235 Westlake Center #381\nDaly City, CA  94015\n650-992-1191          650-504-3161\n\nCustomer:\nContact:\nPhone:\nLocation:\n\nWon Living Trust\nMichelle An\n(415) 713-2018\n1059 M'